# Neural networks as probability models

* [1. Regression (xkcd dataset)](#1.-Regression-/-xkcd-dataset)
* [2. Classification (MNIST dataset)](#2.-Classification-/-MNIST-dataset)
* [3. Naive latent-variable generation (2d point dataset)](#3.-Latent-variable-generation-/-2D-points-dataset)
* [4. Variational autoencoder (MNIST dataset)](#4.-Variational-autoencoder-/-MNIST-dataset)
* [5. Recurrent neural network generation (English names dataset)](#5.-Generation-with-RNN-/-dataset-of-English-names)

In [ ]:
import numpy as np
import pandas
import matplotlib.pyplot as plt
import matplotlib
import torch
import torch.optim as optim
import torch.nn as nn
import torchvision
import IPython

In [ ]:
# Utilities to streamline interactive start-and-stop optimization.

# When I'm training a neural network, it's nice to be able to interrupt training,
# inspect, tweak, and resume training. This class lets me interrupt cleanly
# using the Kernel | Interrupt menu option. (To be precise, it lets me ensure
# that my code is only interrupted at the end of a loop, not in the middle.)
# See the examples below for how it's used.

import signal
class Interruptable():
    class Breakout(Exception):
        pass
    def __init__(self):
        self.interrupted = False
        self.orig_handler = None
    def __enter__(self):
        self.orig_handler = signal.getsignal(signal.SIGINT)
        signal.signal(signal.SIGINT, self.handle)
        return self.check
    def __exit__(self, exc_type, exc_val, exc_tb):
        signal.signal(signal.SIGINT, self.orig_handler)
        if exc_type == Interruptable.Breakout:
            print(' stopped')
            return True
        return False
    def handle(self, signal, frame):
        if self.interrupted:
            self.orig_handler(signal, frame)
        print('Interrupting ...', end='')
        self.interrupted = True
    def check(self):
        if self.interrupted:
            raise Interruptable.Breakout

# If I interrupt training, and then resume, it's nice to be able to resume
# exactly where I left off. This function defines a Python generator which
# cycles through the items a dataset over and over again, and because it's a
# generator it can keep track of which epoch it's on.
            
def enumerate_cycle(g, shuffle=True):
    epoch = 0
    while True:
        if shuffle:
            for i,j in enumerate(np.random.permutation(len(g))):
                yield (epoch,i), g[j]
        else:
            for i,x in enumerate(g):
                yield (epoch,i), x
        epoch = epoch + 1

# 1. Regression / xkcd dataset

In [ ]:
# The data behind the comic https://xkcd.com/2048/
url = 'https://www.cl.cam.ac.uk/teaching/2223/DataSci/data/xkcd.csv'
xkcd = pandas.read_csv(url)
plt.scatter(xkcd.x, xkcd.y)
plt.show()

In [ ]:
# Model: Y_i ~ ?(x_i) + N(0,?^2)
# The ? function is a messy neural network, specified in the Wiggle class.

class Wiggle(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(
            nn.Linear(1,4),
            nn.LeakyReLU(),
            nn.Linear(4,20),
            nn.LeakyReLU(),
            nn.Linear(20,20),
            nn.LeakyReLU(),
            nn.Linear(20,1)
        )
    def forward(self, x):
        return self.f(x)


class RWiggle(nn.Module):
    def __init__(self):
        super().__init__()
        self.? = Wiggle()
        self.? = nn.Parameter(torch.tensor(1.0))
        
    # Compute log Pr(y;x) for a batch of x,y data
    def forward(self, y, x):
        ?2 = torch.pow(self.?, 2)
        return - 0.5*torch.log(2*np.pi*?2) - torch.pow(y - self.?(x), 2) / (2*?2)

In [ ]:
x,y = torch.tensor(xkcd.x, dtype=torch.float)[:,None], torch.tensor(xkcd.y, dtype=torch.float)[:,None]

rwiggle = RWiggle()
epoch = 0

In [ ]:
optimizer = optim.Adam(rwiggle.parameters())

with Interruptable() as check_interrupted:
    while True:
        check_interrupted()
        optimizer.zero_grad()
        loglik = torch.mean(rwiggle(y, x))
        (-loglik).backward()
        optimizer.step()
        epoch += 1

        if epoch % 200 == 0:
            IPython.display.clear_output(wait=True)
            xnew = torch.linspace(.8,9.2,50)[:,None]
            with torch.no_grad():
                ynew = rwiggle.?(xnew).detach()[:,0]
                ? = rwiggle.?.detach()
            ax = plt.gca()
            ax.fill_between(xnew[:,0], ynew-2*?, ynew+2*?, color='steelblue', alpha=.6)
            ax.plot(xnew, ynew, color='steelblue')
            plt.scatter(x, y, color='black', marker='+', alpha=.8)
            print(f'epoch={epoch} loglik={loglik.item():.4} ?={rwiggle.?.item():.4}')
            plt.show()

# 2. Classification / MNIST dataset

In [ ]:
# PyTorch comes with a simple wrapper around the MNIST dataset.
# The dataset consists of a pair of (numpy_array, int) representing images and labels.
# We can pull out (image,label) pairs using e.g.
#   img,lbl = mnist[0]
#   img,lbl = next(iter(mnist))

mnist = torchvision.datasets.MNIST(
    root = 'pytorch-data/',    # where to put the files
    download = True,   # if files aren't here, download them
    train = True,      # whether to import the test or the train subset
    # PyTorch uses PyTorch tensors internally, not numpy arrays, so convert them.
    transform = torchvision.transforms.ToTensor()
)


# Very many PyTorch building blocks assume that the data comes in batches.
# The DataLoader converts the full mnist list [(img,lbl),...] into batches
#     [(img_batch,lbl_batch), ...]
# where each img_batch is an array with an extra dimension prepended.

mnist_batched = torch.utils.data.DataLoader(mnist, batch_size=5)

In [ ]:
# Look at a few sample images

import collections
img_i = collections.defaultdict(list)
for i,(_,k) in enumerate(mnist):
    img_i[k].append(i)
    
rng = np.random.default_rng()
show_i = [rng.choice(img_i[k]) for k in [1,6,1,8,0,3]]
show = torch.stack([mnist[i][0] for i in show_i])
x = torchvision.utils.make_grid(show, nrow=6, pad_value=1)
plt.imshow(x.numpy().transpose((1,2,0)))
plt.xticks([])
plt.yticks([])
plt.axis('off')
plt.show()
show_i

In [ ]:
class MyImageClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(       # input shape [B*1*28*28]
            nn.Conv2d(1, 32, 3, 1),   # -> [B*32*26*26]
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, 1),  # -> [B*64*24*24]
            nn.MaxPool2d(2),          # -> [B*64*12*12]
            nn.Dropout2d(0.25),
            nn.Flatten(1),            # -> [B*9216]
            nn.Linear(9216, 128),     # -> [B*128]
            nn.ReLU(),
            nn.Dropout2d(0.5),
            nn.Linear(128, 10)        # -> [B*10]
        )

    # compute log likelihood for a batch of data
    def forward(self, y, x):          # x.shape [B*1*28*28], y.shape [B], output.shape [B]
        return - nn.functional.cross_entropy(self.f(x), y, reduction='none')

    # compute the class probabilities for a single image
    def classify(self, x):  # input: [1*28*28] array
        q = self.f(torch.as_tensor(x)[None,...])[0]
        return nn.functional.softmax(q, dim=0)

In [ ]:
model = MyImageClassifier()
epoch = 0

In [ ]:
model.train(mode=True) # put it into training mode (turns dropout on)
optimizer = optim.Adam(model.parameters())

with Interruptable() as check_interrupted:
    for batch_num, (imgs,lbls) in enumerate(mnist_batched):
        check_interrupted()
        optimizer.zero_grad()
        loglik = model(lbls, imgs)
        e = - torch.mean(loglik)
        e.backward()
        optimizer.step()

        if batch_num % 25 == 0:
            IPython.display.clear_output(wait=True)
            print(f'epoch={epoch} batch={batch_num}/{len(mnist_batched)} loglik={-e.item()}')
    epoch += 1

In [ ]:
model.train(mode=False)  # put it into execution mode (turns dropout off)
img,lbl = mnist[14028]
p = model.classify(img).detach().numpy()

fig,(ax1,ax2) = plt.subplots(1,2)
ax1.imshow(img[0], cmap='gray')
ax2.barh(np.arange(10), p)
ax2.set_yticks(np.arange(10))
ax1.set_title(f'Official label {lbl}')
plt.show()

# 3. Latent-variable generation / 2D points dataset

In [ ]:
# Randomly jittered points from MNIST image 43876
url = 'https://www.cl.cam.ac.uk/teaching/2223/DataSci/data/digit8.csv'
digit8 = pandas.read_csv(url)
fig,ax = plt.subplots()
ax.scatter(digit8.x1, digit8.x2, marker='+', color='black', alpha=.7)
ax.set_aspect('equal')
ax.set_xticks([])
ax.set_yticks([])
plt.show()

In [ ]:
class RCurve(nn.Module):
    def __init__(self, m=100):
        super().__init__()
        self.f = nn.Sequential(
            nn.Linear(1,4),
            nn.LeakyReLU(),
            nn.Linear(4,20),
            nn.LeakyReLU(),
            nn.Linear(20,20),
            nn.LeakyReLU(),
            nn.Linear(20,2)
            )
        self.? = nn.Parameter(torch.tensor(0.1)) # initial guess: ?=0.1
        self.z = torch.linspace(0,1,m)
    
    # The forward function computes the log likelihood of a batch of coordinates
    def forward(self, x):
        # x is a batch of 2d coordinates, i.e. shape batchsize*2
        ? = self.f(self.z.reshape(-1,1)).reshape(1,-1,2)
        x = x.reshape(-1,1,2)
        d = torch.linalg.norm(x - ?, dim=2)
        lik = torch.exp(- 0.5 * torch.pow(d/self.?, 2))
        lik = torch.log(torch.mean(lik, dim=1)) - torch.log(2*np.pi*torch.pow(self.?,2))
        return lik # returns a batch of loglik values

In [ ]:
X = torch.as_tensor(digit8.values)
mymodel = RCurve()
epoch = 0

In [ ]:
optimizer = optim.Adam(mymodel.parameters())

with Interruptable() as check_interrupted:
    while True:
        check_interrupted()
        optimizer.zero_grad()
        loglik = mymodel(X)
        e = - torch.mean(loglik)
        e.backward()
        optimizer.step()

        epoch += 1
        if epoch % 200 == 0:
            IPython.display.clear_output(wait=True)
            print(f'epoch={epoch} loglik={-e.item():.4} ?={mymodel.?.item():.4}')
            z = np.linspace(0,1,101)
            zt = torch.tensor(z, dtype=torch.float).reshape(-1,1)
            pred = mymodel.f(zt).detach()
            ? = mymodel.?.item()
            circles = [plt.Circle(?, 2*?) for ? in pred]
            circles = matplotlib.collections.PatchCollection(circles, color='steelblue', alpha=.1, linewidth=None)
            plt.gca().add_collection(circles)
            plt.plot(pred[:,0], pred[:,1], color='goldenrod', linewidth=4)
            plt.scatter(X[:,0], X[:,1], c='black')
            plt.gca().set_aspect('equal')
            plt.show()

# 4. Variational autoencoder / MNIST dataset

In [ ]:
# Using the MNIST dataset

import torchvision

mnist = torchvision.datasets.MNIST(
    root = 'pytorch-data/',    # where to put the files
    download = True,   # if files aren't here, download them
    train = True,      # whether to import the test or the train subset
    # PyTorch uses PyTorch tensors internally, not numpy arrays, so convert them.
    transform = torchvision.transforms.ToTensor()
)

show = [mnist[i] for i in [59289, 28001, 35508, 43876, 23627, 14028]]
show = torch.stack([img for img,lbl in show])
x = torchvision.utils.make_grid(show, nrow=6, pad_value=1)
plt.imshow(x.numpy().transpose((1,2,0)))
plt.axis('off')
plt.show()

mnist_batched = torch.utils.data.DataLoader(mnist, batch_size=100)

In [ ]:
# Generator model:
# Given a latent variable z ? R^d, compute a 28?28 matrix p(z), and then
# let each pixel be generated by X_{i,j} ~ Bin(1, p_{i,j}(z)).
# (Or, if we want to be fancy, let's say X_{i,j} is a Continuous Bernoulli random
# variable with parameter p_{i,j}(z).)
#
# Encoder model:
# Given a 28?28 greyscale image x, compute ? and log(?^2), both ? R^d,
# and let the encoding be: Ztilde ~ N(?, ?^2).


class BernoulliImageGenerator(nn.Module):
    def __init__(self, d=4):
        super().__init__()
        self.d = d
        self.p = nn.Sequential(
            nn.Linear(d, 128),
            nn.LeakyReLU(),
            nn.Linear(128, 1728),
            nn.LeakyReLU(),
            nn.Unflatten(1, (12,12,12)), # -> B?12?12?12
            nn.Conv2d(12, 36, 3, 1),     # -> [B?36?10?10]
            nn.LeakyReLU(),
            nn.Flatten(1),               # -> [B?3600]
            nn.Unflatten(1, (4,30,30)),  # -> [B?4?30?30]
            nn.Conv2d(4, 4, 3, 1),       # -> [B?4?28?28]
            nn.LeakyReLU(),
            nn.Conv2d(4, 1, 1, 1),       # -> [B?1?28?28]
            nn.Sigmoid()
        )
    def forward(self, z):
        return self.p(z)
    def loglik(self, x, z):
        xr = self(z)
        return (x*torch.log(xr) + (1-x)*torch.log(1-xr)).sum((1,2,3))
    

class GaussianEncoder(nn.Module):
    def __init__(self, decoder):
        super().__init__()
        self.d = decoder.d
        self.decode = decoder
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1),
            nn.LeakyReLU(),
            nn.Conv2d(32, 64, 3, 1),
            nn.MaxPool2d(2),
            nn.Flatten(1),
            nn.Linear(9216, 128),
            nn.LeakyReLU(),
            nn.Linear(128, self.d*2)
        )

    # Given an image x, compute (?,?)
    def forward(self, x):
        ?? = self.enc(x)
        ?,? = ??[:,:self.d], ??[:,self.d:]
        return ?, torch.exp(?/2)

    # The log likelihood lower bound
    def loglik_lb(self, x):
        ?,? = self(x)
        kl = 0.5 * (?**2 + ?**2 - torch.log(?**2) - 1).sum(1)
        ? = torch.randn_like(?)
        ll = self.decode.loglik(x, z=?+?*?)
        return ll - kl

In [ ]:
gen = BernoulliImageGenerator(d=4)
model = GaussianEncoder(gen)

ei_mnist = enumerate_cycle(mnist_batched, shuffle=False)

In [ ]:
optimizer = optim.Adam(model.parameters())
zsamp = torch.normal(mean=0.0, std=1.0, size=(6,model.d))

with Interruptable() as check_interrupted:
    for (epoch,batch_num),(imgs,lbls) in ei_mnist:
        check_interrupted()
        optimizer.zero_grad()
        loglik_lb = torch.mean(model.loglik_lb(imgs))
        (-loglik_lb).backward()
        optimizer.step()

        if batch_num % 25 == 0:
            IPython.display.clear_output(wait=True)
            print(f'epoch={epoch} batch={batch_num}/{len(mnist_batched)} loglik={loglik_lb.item()}')
            with torch.no_grad():
                rx = model.decode(zsamp)
                imgs = torchvision.utils.make_grid(rx, nrow=7, pad_value=1)
            plt.imshow(imgs.numpy().transpose((1,2,0)))
            plt.axis('off')
            plt.show()

# 5. Generation with RNN / dataset of English names

In [ ]:
url = "https://www.cl.cam.ac.uk/teaching/2122/DataSci/data/english_names.txt"
names = pandas.read_table(url, header=None, names=['X']).X.str.lower()

names.sample(5)

In [ ]:
# The neural network recurrent unit (called f_? in lecture notes) takes in a character
# and a hidden state. In order for a neural network to accept a character, the character
# has to be encoded into a number or vector. Here we're creating one-hot coded versions
# of the names: each item in names_oh is an array with shape (name_len ? alphabet_len),
#
#      names_oh[i][l,a] = Indicator[name i has character a in position l]

alphabet = list(set(''.join(names)))
alphabet_n = {c:i+1 for i,c in enumerate(alphabet)}
A = len(alphabet)+1

names_n = [[0] + [alphabet_n[x] for x in name] + [0] for name in names]
names_n = [torch.tensor(n) for n in names_n]
names_oh = [nn.functional.one_hot(n, num_classes=A).float() for n in names_n]

names_oh,names_oh_holdout = names_oh[:3000], names_oh[3000:]

In [ ]:
# A simple model for a random character sequence.
# See printed lecture notes for an explanation.

class RSeq(nn.Module):
    def __init__(self, H=50, L=2):
        super().__init__()
        self.rnn = nn.GRU(input_size=A, hidden_size=H, num_layers=L, dropout=.05)
        self.map = nn.Linear(self.rnn.hidden_size, A)
    
    def f(self, x, v=None):  # x.shape=m*b*A, v.shape=L*b*H
        assert x.shape[1]==1, "This code only works with batch_size=1"
        y,v = self.rnn(x, v) # y.shape=m*b*H, v.shape=L*b*H
        z = self.map(y)      # z.shape=m*b*A
        logp = nn.functional.log_softmax(z, dim=2) # logp.shape=m*b*A
        return v,logp
    
    def forward(self, x): # x.shape = (n+1)*b*A
        _,logp = self.f(x[:-1])      # logp.shape = n*b*A
        return torch.sum(logp * x[1:], dim=(0,2)) # shape = b
    
    def generate(self):
        is_training = self.training
        self.train(False)
        with torch.no_grad():
            res = ''
            x_n,v = 0,None
            while True:
                x_n = torch.tensor(x_n).reshape(1,1)
                x_oh = nn.functional.one_hot(x_n, num_classes=A).float()
                v,logp = model.f(x_oh, v)
                p = torch.exp(logp[0,0])
                x_n = np.random.choice(A, p=p.detach().numpy())
                if x_n == 0: break
                res += alphabet[x_n-1]
        self.train(is_training)
        return res

In [ ]:
model = RSeq()
model.train(True)
optimizer = optim.Adam(model.parameters())
ei_names = enumerate_cycle(np.random.permutation(len(names_oh)))
loss_history = []

In [ ]:
with Interruptable() as check_interrupted:
    for (epoch,i),j in ei_names:
        check_interrupted()
        optimizer.zero_grad()
        loglik = model(names_oh[j][:,None,:])
        (-loglik).backward()
        optimizer.step()

        if i % 20 == 0:
            IPython.display.clear_output(wait=True)
            with torch.no_grad():
                eval_loglik = np.mean([model(n[:,None,:]).item() for n in names_oh_holdout])
            print(f'epoch={epoch} item={i}/{len(names_oh)} val.loglik={eval_loglik.item():.4} generated_name={model.generate()}')
            loss_history.append((epoch+i/len(names_oh), loglik.item(), eval_loglik))

In [ ]:
# Simple plot of the training loss

df = pandas.DataFrame.from_records(loss_history, columns=['epoch','loglik_tr','loglik_eval'])
fig,ax = plt.subplots(figsize=(6,2))
ax.plot(df.epoch, df.loglik_tr.ewm(com=8).mean(), linewidth=1, label='training') # ewm computes moving average
ax.plot(df.epoch, df.loglik_eval, linewidth=3, linestyle='dashed', color='black', label='validation')
ax.axhline(y=np.log(1/len(names_oh)), color='black', linestyle=':')

ax.set_ylim([-30,-5])
ax.set_title('log likelihood per datapoint', fontsize=14)
ax.set_xlabel('epoch', fontsize=14)
plt.legend(bbox_to_anchor=(1,.5), loc='center left', frameon=False, fontsize=14)
plt.show()